# Content Understanding tables — EN 10204 material certificates

Active notebook **02**. Uses Azure Content Understanding `prebuilt-layout` to extract tables as structured cells (row/column indices + `kind`).

This is the path to use when you need **spec (Min/Max) vs result** rows for chemistry / mechanical tables.
For OCR text + field JSON via Mistral, see `notebooks/01-mistral-ocr-and-fields.ipynb`.

**Outputs (under `extraction-output/`):**
- `cu-layout/<cert>.json` — full API layout result
- `cu-tables/<cert>.json` — tables with headers / spec_rows / result_rows
- `cu-reports/<cert>.html` — human view by page (written on every `save_layout`; no API)

See repo `README.md` for the full flow.


## Setup

In [ ]:
import json
import os
import re
import sys
from pathlib import Path

from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import AnalysisInput
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv
from IPython.display import Markdown, display

def find_repo_root() -> Path:
    """Resolve repo root whether the kernel cwd is repo root or notebooks/."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "raw-docs").is_dir() and (candidate / "scripts").is_dir():
            return candidate
    return here

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

ENDPOINT    = os.getenv("AZURE_CONTENT_UNDERSTANDING_ENDPOINT", "")
KEY         = os.getenv("AZURE_CONTENT_UNDERSTANDING_KEY", "")
API_VERSION = "2025-11-01"
ANALYZER    = "prebuilt-layout"

INPUT_DIR   = ROOT / "raw-docs"
OUTPUT_DIR  = ROOT / "extraction-output"
LAYOUT_DIR  = OUTPUT_DIR / "cu-layout"    # full Content Understanding layout JSON
TABLES_DIR  = OUTPUT_DIR / "cu-tables"    # parsed headers / spec / result rows
REPORTS_DIR = OUTPUT_DIR / "cu-reports"   # HTML for human review (local, no API)

for folder in [LAYOUT_DIR, TABLES_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

client = ContentUnderstandingClient(
    endpoint=ENDPOINT,
    credential=AzureKeyCredential(KEY),
    api_version=API_VERSION
)

print("Repo root:", ROOT)
print("Endpoint :", ENDPOINT or "⚠  not set — add AZURE_CONTENT_UNDERSTANDING_ENDPOINT to .env")
print("Key set  :", bool(KEY))
print("Analyzer :", ANALYZER)


## Layout extraction

In [2]:
def extract_layout(pdf_path: Path) -> dict:
    """
    Send a local PDF to Azure Content Understanding prebuilt-layout analyzer.
    Returns the full analysis result as a dict.

    The layout analyzer extracts:
    - pages (dimensions, words, lines)
    - tables (cells with row/column indices, bounding boxes, content)
    - paragraphs

    Tables are the critical output for EN 10204 — each cell knows its exact
    row and column position, so spec rows and result rows can be told apart.
    """
    pdf_bytes = pdf_path.read_bytes()
    poller = client.begin_analyze(
        analyzer_id=ANALYZER,
        inputs=[AnalysisInput(data=pdf_bytes, mime_type="application/pdf")]
    )
    result = poller.result()
    return result.as_dict()

## Table parsing

The layout result (`contents[0]['tables']`) contains tables as flat lists of cells.
Each cell has:
- `rowIndex`, `columnIndex` — position in the table
- `content` — text inside the cell
- `kind` — `'columnHeader'` for header cells, `'content'` for data cells

We reconstruct each table as a 2D grid, then classify each data row as:
- **spec** — Min/Max limits from the specification (first cell: Req't, Min, Max, etc.)
- **result** — actual measured test values for the supplied heat (first cell: Result, heat ID, etc.)

In [3]:
def _cells_to_grid(cells: list[dict], row_count: int, col_count: int) -> list[list[str]]:
    """Reconstruct a 2D grid from the flat cell list returned by the API."""
    grid = [[""] * col_count for _ in range(row_count)]
    for cell in cells:
        row_index = cell.get("rowIndex", 0)
        column_index = cell.get("columnIndex", 0)
        if row_index < row_count and column_index < col_count:
            grid[row_index][column_index] = cell.get("content", "").strip()
    return grid


# Keywords that identify the first cell of a spec (limit) row.
_SPEC_KEYWORDS = re.compile(
    r"^\s*(req|min|max|limit|spec|minimum|maximum|shall|required)",
    re.IGNORECASE
)

# Keywords that identify the first cell of a result (measured value) row.
_RESULT_KEYWORDS = re.compile(
    r"^\s*(result|actual|achieved|measured|test\s*value)",
    re.IGNORECASE
)

# First cell looks like a heat number (alphanumeric id, 3 or more characters).
_HEAT_ID_RE = re.compile(r"^[A-Z0-9]{3,}", re.IGNORECASE)


def _classify_row(row: list[str]) -> str:
    """
    Classify a data row as "spec" (specification limit) or "result" (measured value).

    Rules applied in order:
    1. First cell matches a spec keyword (Req't, Min, Max, Limit...) -> spec
    2. First cell is empty AND remaining non-empty cells are all numeric -> spec
       (handles unlabeled Min rows where the row-label column has a merged cell above,
       e.g. MELESI DH44 chemistry table)
    3. First cell matches a result keyword (Result, Achieved, Actual...) -> result
    4. First cell looks like a heat number (alphanumeric ID >= 3 chars) -> result
    5. Default -> result
    """
    first = row[0] if row else ""

    if _SPEC_KEYWORDS.search(first):
        return "spec"

    if (not first.strip()
            and any(re.search(r"^\d", cell) for cell in row[1:] if cell)
            and not any(re.search(r"[A-Za-z]", cell) for cell in row[1:] if cell)):
        return "spec"

    if _RESULT_KEYWORDS.search(first):
        return "result"

    if _HEAT_ID_RE.match(first):
        return "result"

    return "result"


def parse_tables(layout_result: dict) -> list[dict]:
    """
    Extract and classify tables from the API's structured output.

    Uses contents[0]['tables'] — the structured array returned by prebuilt-layout.
    Each cell has rowIndex, columnIndex, content, and kind.
    kind='columnHeader' marks header cells; kind='content' marks data cells.

    A row is a header row if any of its cells have kind='columnHeader'.
    Remaining rows are classified as spec or result by _classify_row().
    """
    api_tables = layout_result.get("contents", [{}])[0].get("tables", [])

    tables = []
    for table in api_tables:
        row_count = table.get("rowCount", 0)
        col_count = table.get("columnCount", 0)
        cells = table.get("cells", [])

        grid = _cells_to_grid(cells, row_count, col_count)

        header_row_indices = {
            cell.get("rowIndex", 0)
            for cell in cells
            if cell.get("kind") == "columnHeader"
        }

        headers     = []
        spec_rows   = []
        result_rows = []

        for row_idx, row in enumerate(grid):
            if row_idx in header_row_indices:
                headers.append(row)
            elif _classify_row(row) == "spec":
                spec_rows.append(row)
            else:
                result_rows.append(row)

        tables.append({
            "headers":     headers,
            "spec_rows":   spec_rows,
            "result_rows": result_rows,
        })

    return tables


## Save helpers

In [4]:
def save_layout(pdf_path: Path, layout: dict) -> Path:
    out = LAYOUT_DIR / f"{pdf_path.stem}.json"
    out.write_text(json.dumps(layout, indent=2, ensure_ascii=False), encoding="utf-8")
    try:
        from scripts.layout_json_to_html import convert_file

        convert_file(out, REPORTS_DIR)
    except Exception as e:
        print("HTML report skipped:", e)
    return out


def save_tables(pdf_path: Path, tables: list[dict]) -> Path:
    out = TABLES_DIR / f"{pdf_path.stem}.json"
    out.write_text(json.dumps(tables, indent=2, ensure_ascii=False), encoding="utf-8")
    return out

## List PDFs

In [5]:
pdf_files = sorted(INPUT_DIR.glob("*.pdf"))
for i, p in enumerate(pdf_files):
    print(f"[{i}] {p.name}  ({p.stat().st_size // 1024} KB)")

[0] 147149__02 MELESI MA-00351 mds-DH44.pdf  (5179 KB)
[1] EUR-472273.pdf  (4135 KB)
[2] FMT-22-1454-EZ17.pdf  (1499 KB)


## Process one file

In [ ]:
FILE_INDEX = 0 #change to which file you want to process!

target = pdf_files[FILE_INDEX]
print(f"Analyzing: {target.name}")

layout = extract_layout(target)
tables = parse_tables(layout)

layout_path = save_layout(target, layout)
tables_path = save_tables(target, tables)

print(f"Tables found : {len(tables)}")
print(f"Layout saved : {layout_path.relative_to(OUTPUT_DIR)}")
print(f"Tables saved : {tables_path.relative_to(OUTPUT_DIR)}")
print(f"HTML report  : {(REPORTS_DIR / (target.stem + '.html')).relative_to(OUTPUT_DIR)}")

Analyzing: 147149__02 MELESI MA-00351 mds-DH44.pdf
Tables found : 26
Layout saved : layout-v1\147149__02 MELESI MA-00351 mds-DH44.json
Tables saved : tables-v1\147149__02 MELESI MA-00351 mds-DH44.json
HTML report  : reports\147149__02 MELESI MA-00351 mds-DH44.html


## Inspect extracted tables

For each table, shows headers, spec rows (Min/Max limits), and result rows (actual test values) separately.

In [ ]:
#only inline print, you can skip it!

for table_index, table in enumerate(tables):
    display(Markdown(f"### Table {table_index + 1}"))

    if table["headers"]:
        display(Markdown("**Headers (column names):**"))
        for row in table["headers"]:
            print(" | ".join(row))

    if table["spec_rows"]:
        display(Markdown("**Spec rows (Min/Max limits from specification):**"))
        for row in table["spec_rows"]:
            print(" | ".join(row))

    if table["result_rows"]:
        display(Markdown("**Result rows (actual measured test values):**"))
        for row in table["result_rows"]:
            print(" | ".join(row))

    print()

### Table 1

**Headers (column names):**

N. RIF. | ITEM | N / QUANTITY | DESCRIZIONE / DESCRIPTION / DÉSIGNATION | NOTE | 


**Result rows (actual measured test values):**

SR1852 | 20 | 2 | F51/F60
PREMACHINED RING | mm Øe473xØi300xH145
QTR Nr. 1 / MDS D44 / MS05 | A1126
SR1870 | 40 | 2 | F51/F60
PREMACHINED RING | mm Øe473xØi174xH153 | A1008
 |  |  |  | QTR Nr. 1 / MDS D44 / MS05 | 
SR1906 | 60 | 2 | F51/F60
PREMACHINED RING | mm Øe471xØi304xH78
QTR Nr. 1 / MDS D44 / MS05 | A1261



...

**Result rows (actual measured test values):**

CHEMICAL LADLE ANALYSIS ACCORDING TO ASTM A751 - EN ISO 14284 |  |  |  |  |  |  |  |  |  |  | 
Control lot | No. - Weight : 020000944860 | - | 83.888 | KG | - | 184.938 | LB |  |  |  | 
ELEMENTS |  | C | Si | Mn | P | S | N | Al | Cr | Mo | Ni
MEASUREMENT | UNIT | % w/w | % w/w | % w/w | % w/w | % w/w | % w/w | % w/w | % w/w | % w/w | % w/w
OBTAINED |  | 0,014 | 0,39 | 1,18 | 0,026 | 0,0009 | 0,18 | 0,006 | 22,47 | 3,24 | 5,71
ELEMENTS |  | Cu | Sn | Co | B | CV F8 |  |  |  |  | 
MEASUREMENT | UNIT | % w/w | % w/w | % w/w | % w/w | % w/w |  |  |  |  | 
OBTAINED |  | 0,26 | 0,006 | 0,13 | 0,0023 | 36 |  |  |  |  | 
CV F8 = | CR+3.3*MO+16*N2 |  |  |  |  |  |  |  |  |  | 



### Table 26

**Result rows (actual measured test values):**

Control lot | No. - Weight : 020000987188 |  | 79.241 | KG |  | 174.694 | LB |  |  |  | 
ELEMENTS |  | C | Si | Mn | P | S | N | Al | Cr | Mo | Ni
MEASUREMENT | UNIT | % w/w | % w/w | % w/w | % w/w | % w/w | % w/w | % w/w | % w/w | % w/w | % w/w
OBTAINED |  | 0,024 | 0,37 | 1,21 | 0,025 | 0,0005 | 0,18 | 0,006 | 22,46 | 3,17 | 5,62
ELEMENTS |  | Cu | Sn | Co | B | CV F8 |  |  |  |  | 
MEASUREMENT | UNIT | % w/w | % w/w | % w/w | % w/w | % w/w |  |  |  |  | 
OBTAINED |  | 0,21 | 0,005 | 0,14 | 0,0022 | 36 |  |  |  |  | 
CV F8 = | CR+3.3*MO+16*N2 |  |  |  |  |  |  |  |  |  | 



## Batch — process all PDFs
Run only when you want to process all files.

In [ ]:
for pdf in pdf_files:
    print(f"\n{pdf.name}")
    try:
        layout = extract_layout(pdf)
        tables = parse_tables(layout)
        save_layout(pdf, layout)
        save_tables(pdf, tables)
        print(f"  {len(tables)} tables → {LAYOUT_DIR.name}/, {TABLES_DIR.name}/, {REPORTS_DIR.name}/")
    except Exception as e:
        print(f"  ERROR: {e}")

## Layout JSON → HTML (optional bulk rebuild)

Each `save_layout` already writes `REPORTS_DIR/<stem>.html` for that PDF (no API).

Run the cell below only to regenerate **all** HTML files from every JSON already in `cu-layout/` (e.g. after editing the script).

CLI: `python scripts/layout_json_to_html.py`


In [ ]:
import subprocess
import sys

html_script = ROOT / "scripts" / "layout_json_to_html.py"
if not html_script.is_file():
    raise FileNotFoundError(html_script)
subprocess.run(
    [sys.executable, str(html_script), "--layout-dir", str(LAYOUT_DIR), "--out-dir", str(REPORTS_DIR)],
    check=True,
)
for html_path in sorted(REPORTS_DIR.glob("*.html")):
    print(html_path)
